# SpaceX Falcon 9 — Data Collection with Web Scraping

This notebook scrapes the Wikipedia page **"List of Falcon 9 and Falcon Heavy launches"**
with `BeautifulSoup` and parses the launch table (flight number, date/time, booster
version, launch site, payload, payload mass, orbit, customer, outcome, booster
landing) into a Pandas DataFrame.

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import unicodedata

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
WIKI_URL = "https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches"

resp = requests.get(WIKI_URL, headers=HEADERS, timeout=20)
print("status:", resp.status_code, " bytes:", len(resp.text))
soup = BeautifulSoup(resp.text, "html.parser")
print(soup.title.string)

status: 200  bytes: 2996764


List of Falcon 9 and Falcon Heavy launches - Wikipedia


In [2]:
# Wikipedia's launch tables carry class "wikitable plainrowheaders ..." — the exact
# suffix (collapsible / sticky-header) changes over time, so match robustly on the
# two class names that have stayed constant.
launch_tables = soup.find_all("table", class_=lambda c: c and "wikitable" in c and "plainrowheaders" in c)
print("launch tables found:", len(launch_tables))

def extract_column_from_header(row):
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    colname = ' '.join(row.contents)
    if not colname.strip().isdigit():
        colname = unicodedata.normalize("NFKD", colname).strip()
        return colname

column_names = []
for th in launch_tables[0].find_all('th'):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)
print(column_names)

launch tables found: 2
['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


In [3]:
def date_time(table_cells):
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    out = ''.join([bv for i, bv in enumerate(table_cells.strings) if i % 2 == 0][0:-1])
    return out

def landing_status(table_cells):
    return [i for i in table_cells.strings][0]

def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass = mass[0:mass.find("kg") + 2]
    else:
        mass = 0
    return mass

extracted_row = 0
launch_dict = {
    'Flight No.': [], 'Date': [], 'Time': [], 'Version Booster': [], 'Launch site': [],
    'Payload': [], 'Payload mass': [], 'Orbit': [], 'Customer': [], 'Launch outcome': [],
    'Booster landing': [],
}

for table in launch_tables:
    for rows in table.find_all("tr"):
        th = rows.find('th')
        flag = bool(th and th.get_text(strip=True).isdigit())
        row = rows.find_all('td')
        if flag and len(row) >= 8:
            extracted_row += 1
            launch_dict['Flight No.'].append(th.get_text(strip=True))
            datatimelist = date_time(row[0])
            launch_dict['Date'].append(datatimelist[0].strip(',') if datatimelist else None)
            launch_dict['Time'].append(datatimelist[1] if len(datatimelist) > 1 else None)
            bv = booster_version(row[1])
            if not bv:
                bv = row[1].a.string if row[1].a else row[1].get_text(strip=True)
            launch_dict['Version Booster'].append(bv)
            launch_dict['Launch site'].append(row[2].a.string if row[2].a else row[2].get_text(strip=True))
            launch_dict['Payload'].append(row[3].a.string if row[3].a else row[3].get_text(strip=True))
            launch_dict['Payload mass'].append(get_mass(row[4]))
            launch_dict['Orbit'].append(row[5].a.string if row[5].a else row[5].get_text(strip=True))
            launch_dict['Customer'].append(row[6].a.string if row[6].a else row[6].get_text(strip=True))
            launch_dict['Launch outcome'].append(list(row[7].strings)[0] if row[7].strings else None)
            launch_dict['Booster landing'].append(landing_status(row[8]) if len(row) > 8 else None)

print("Parsed", extracted_row, "table rows from the Wikipedia launch tables.")

Parsed 262 table rows from the Wikipedia launch tables.


In [4]:
df = pd.DataFrame({k: v for k, v in launch_dict.items() if len(v) == extracted_row})
print(df.shape)
df.head(10)

(262, 11)


,Flight No.,Date,Time,Version Booster,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing
0,418,"January 4, 2025",01:27,F9B5,Cape Canaveral,Thuraya 4-NGS,"5,000 kg",GTO,Thuraya,Success,Success (
1,419,"January 6, 2025",20:43,F9B5,Cape Canaveral,Starlink,"~17,500 kg",LEO,SpaceX,Success,Success (
2,420,"January 8, 2025",15:27,F9B5,Kennedy,Starlink,"~16,500 kg",LEO,SpaceX,Success,Success (
3,421,"January 10, 2025",03:53,F9B5,Vandenberg,Starshield,U,LEO,NRO,Success,Success (
4,422,"January 10, 2025",19:11,F9B5,Cape Canaveral,Starlink,"~16,500 kg",LEO,SpaceX,Success,Success (
5,423,"January 13, 2025",16:47,F9B5,Cape Canaveral,Starlink,"~16,500 kg",LEO,SpaceX,Success,Success (
6,424,"January 14, 2025",19:09,F9B5,Vandenberg,Transporter-12,"2,000 kg",SSO,Various,Success,Success (
7,425,"January 15, 2025",06:11,F9B5,Kennedy,Blue Ghost Mission 1,"2,517 kg",TLI,Firefly Aerospace,Success,Success (
8,426,"January 21, 2025",05:24,F9B5,Kennedy,Starlink,"~15,300 kg",LEO,SpaceX,Success,Success (
9,427,"January 21, 2025",15:45,F9B5,Vandenberg,Starlink,"~15,500 kg",LEO,SpaceX,Success,Success (


In [5]:
df.to_csv('spacex_web_scraped.csv', index=False)
print("Saved spacex_web_scraped.csv —", df.shape[0], "rows scraped live from Wikipedia.")
print()
print("Note: downstream notebooks (wrangling / SQL / visualization / ML) use the")
print("IBM Skills Network cleaned+merged extract of this same public source")
print("(Spacex.csv / dataset_part_1.csv) so that every later step in this project")
print("is reproducible from one consistent, versioned dataset.")

Saved spacex_web_scraped.csv — 262 rows scraped live from Wikipedia.

Note: downstream notebooks (wrangling / SQL / visualization / ML) use the
IBM Skills Network cleaned+merged extract of this same public source
(Spacex.csv / dataset_part_1.csv) so that every later step in this project
is reproducible from one consistent, versioned dataset.
